# One-step loss + Jacobian gap in all directions + cosine along the model's own error on the ResNets  (finetune)

**Why.** The gap-shift diagnostic showed that the rollout error follows two things: the one-step mistake
`d = f(u) - P(u)` and the Jacobian gap **along the model's own error** `||(J_f - J_P) e_hat||`. Random probes spend
almost all their effort on high wavenumbers that the physics damps; the gap along the model's own error is 5 to 2000
times larger per direction than in the growth directions. So: keep the one-step loss (it pins `d`) and add a Jacobian
term along the direction the model's own rollout error actually points.

**Loss.** Every update takes the same 32 windows of 20 consecutive true training states `x_0 .. x_19` (targets
`x_1 .. x_20`). The model is rolled out from `x_0` (no gradient through the rollout): `z_0 = x_0`, `z_k = f(z_{k-1})`.
Its own error at `x_k` is `e_k = z_k - x_k`, direction `e_hat_k = e_k / ||e_k||` (treated as fixed, `sg`). With
`a_k = J_f(x_k) e_hat_k`, `b_k = J_P(x_k) e_hat_k` (one forward-mode product each), `D` = numbers per state:

| arm | loss |
|---|---|
| `mse` | `MSE = mean_k ||f(x_k) - x_{k+1}||^2 / D` (control: the original loss, same data, same updates) |
| `ownR` | `MSE + lam * mean_{k>=1} ||a_k - b_k||^2 / D` |
| `owncosR` | `MSE + lam * mean_{k>=1} [ (||a_k|| - ||b_k||)^2 + BETA_COS ||b_k||^2 (1 - cos(a_k, b_k)) ] / D` |
| `randR` | `MSE + lam * mean_{k>=1} ||(J_f - J_P) v_k||^2 / D`, `v_k` a fresh random unit direction (strategy S2) |
| `bothR` | `MSE + lam_r * (random term) + lam_o * (own-error term)` |
| `cosC` | `MSE + lam_c * mean_{k>=1} (1 - cos(a_k, b_k))`: only the direction in which the own error is carried |
| `randcosR_C` | `MSE + lam_r * (random term, ratio R) + lam_c * (1 - cos(a_k, b_k)) (ratio C)`: the gap closed in every direction, plus the own error carried in the physics' direction so the gap cannot hide there |

`R` (and `C`) is the weight ratio: `lam = R * MSE / term`, both measured on 16 training batches at the moment the term is
switched on, then kept fixed (`R = 100`: the term starts 100 times larger than the one-step loss). Note
`||a - b||^2 = (||a|| - ||b||)^2 + 2 ||a|| ||b|| (1 - cos)`: the plain squared gap already contains a size part and a
direction part; `owncos` weights the direction part `BETA_COS / 2` times more.

**Modes.** `finetune`: start from the dataset's best one-step ResNet of that seed, the term is on from the start.
`scratch`: start from the same random initialisation as the dataset's run of that seed (`PRNGKey(seed)`), one-step
loss only for the first `TERM_ON` of the updates, then the term ramps in over `TERM_RAMP`.

**Checkpoint.** As in the original training, the best checkpoint is the one with the lowest validation rollout
error (16 validation trajectories, 100 steps), for every arm alike (scratch: only after the term is fully on).

**No leakage.** Training windows from the training trajectories only; validation only picks the checkpoint;
test only in the analysis.

**Run.** GPU, Internet on, attach the four-PDE dataset. Runs are ordered PDE by PDE and seed by seed (control
first), so a session that stops early leaves complete comparisons. If the session's time runs out, the next run is
not started: attach this notebook's output and run again to continue. Every validation line must print PASS.

In [ ]:
MODE   = "finetune"                 # "finetune" or "scratch" (the only line that differs between the two notebooks)
PDES   = ["ks_L60", "gks_d0.2", "cgle_phaseL1"]
ARCH   = "resnet"
SEEDS  = [0, 1]
HW, BATCH = 20, 32                  # 32 windows of 20 states per update (640 states), identical for every arm
BETA_COS = 10.0                     # weight of the direction part in arm owncos
CAL_BATCHES = 16                    # training batches used to set lam when the term is switched on
if MODE == "finetune":
    ARMS = ["mse", "rand100", "cos100", "randcos100_10", "randcos100_100", "randcos100_1000"]
    UPDATES, LR, WARMUP, LR_MIN = 20_000, 1e-4, 300, 1e-6
    TERM_ON, TERM_RAMP = 0.0, 0.0
else:
    ARMS = ["mse", "rand100", "own100", "both100"]
    UPDATES, LR, WARMUP, LR_MIN = 60_000, 1e-3, 1000, 1e-5
    TERM_ON, TERM_RAMP = 0.3, 0.1   # fractions of UPDATES: one-step loss only before TERM_ON, then a linear ramp
CAL0, CAL1 = 50, 250                # updates timed to check that the run fits in the session
N_EVAL = 40                         # validation points per run
HOURS, RESERVE_MIN = 11.0, 5.0      # session budget (Kaggle stops at 12 h); minutes kept for each run's analysis
NTRAIN, TRAIN_H, NVAL, VAL_H, NTEST, S, WARM = 128, 50, 16, 100, 30, 200, 300   # exactly the original study
VAL_STRIDE = 10
TEST_EVERY, K_PROBES = 10, 4
NDENSE, TDENSE, KLYAP, CH = 2, 100, 8, 10
VPT_THR, CORR_THR = 0.5, (0.9, 0.8)
import os
SMOKE = False                       # True: tiny runs that only check that everything works
SMOKE = SMOKE or os.environ.get("SMOKE") == "1"
if SMOKE:
    PDES, SEEDS, UPDATES, WARMUP, CAL0, CAL1, N_EVAL = PDES[:1], [0], 120, 10, 10, 40, 4
    HW, BATCH, CAL_BATCHES, NTEST, S, NDENSE, TDENSE = 6, 4, 2, 4, 60, 1, 20
    HOURS, RESERVE_MIN = 3.0, 0.5
    ARMS = (["mse", "rand100", "own100", "owncos100", "both100", "cos100", "randcos100_10"] if MODE == "finetune"
            else ["mse", "own100", "randcos100_10"])

In [ ]:
import glob, json, shutil, subprocess, sys, time, math, re
t_start = time.time()
if os.path.isdir("/kaggle"):
    def gpu_ok():
        r = subprocess.run([sys.executable, "-c", "import jax; d = jax.devices(); assert d[0].platform == 'gpu', d; print(jax.__version__, d)"],
                           capture_output=True, text=True)
        print((r.stdout + r.stderr)[-400:]); return r.returncode == 0
    def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
    if not gpu_ok():
        for spec in ["jax[cuda12]==0.7.2", "jax[cuda12]==0.6.2", "jax[cuda12]==0.4.38"]:
            try: pip("-U", spec)
            except subprocess.CalledProcessError: continue
            if gpu_ok(): break
        else:
            raise RuntimeError("no GPU jax: check Accelerator = GPU and Internet = on")
    import importlib.util
    miss = [m for m in ("equinox", "optax") if not importlib.util.find_spec(m)]
    if miss:
        jv = subprocess.run([sys.executable, "-c", "import jax, jaxlib; print(jax.__version__, jaxlib.__version__)"],
                            capture_output=True, text=True).stdout.split()
        pip(f"jax=={jv[0]}", f"jaxlib=={jv[1]}", *miss)

In [ ]:
%%writefile ccore.py
"""ccore: numpy core for the unified Q1-Q3 study on the four chaotic PDEs (float64).

State vectors are flattened channel-major: u of shape (nch, N) -> u.reshape(-1), index c*N + x.

Notation (one test trajectory, S steps, state dimension D = nch*N):
  u_0..u_S         true states, u_{n+1} = P(u_n)
  d_n = f(u_n) - P(u_n)                           one-step mistake on the true state, n = 0..S-1
  J[n] = J(u_n)                                   model or solver Jacobian at the true state
  e_{n+1} = J[n] e_n + d_n,  e_0 = 0              the ledger (linear predictor)
  Phi(j+1 -> n) = J[n-1] ... J[j+1],  Phi(n -> n) = I,   so  e_n = sum_{j<n} Phi(j+1 -> n) d_j
Steps are indexed n = 1..S in every per-step array (array index n-1), as in the original study.

Every function has a known-answer test in self_test(). Run:  python ccore.py
"""
import numpy as np


# ------------------------------------------------------------------ Q1: ledger and its mismatch
def predict(J, d):
    """e_{n+1} = J[n] e_n + d[n], e_0 = 0.  J (T, D, D), d (T, D) -> (T, D) holding e_1..e_T."""
    T, D = d.shape
    e = np.zeros(D); out = np.zeros((T, D))
    for n in range(T):
        e = J[n] @ e + d[n]; out[n] = e
    return out


def split(pred, e):
    """Per-step mismatch of a predictor, split into size and direction (last axis = state).
    T = |pred-e|/|e|,  r = |pred|/|e|,  M = |r-1|,  P = 2 sin(theta/2) = sqrt(2 - 2 cos),  T^2 = M^2 + r P^2."""
    with np.errstate(all="ignore"):
        ne = np.linalg.norm(e, axis=-1); npr = np.linalg.norm(pred, axis=-1)
        cos = np.sum(pred * e, -1) / (npr * ne)
        cos = np.clip(cos, -1.0, 1.0)
        r = npr / ne
        return dict(T=np.linalg.norm(pred - e, axis=-1) / ne, M=np.abs(r - 1), P=np.sqrt(2 - 2 * cos), r=r, cos=cos)


def gmean(x, axis=0):
    """Geometric mean over an axis, ignoring non-finite and non-positive entries (as the original study)."""
    x = np.where(np.isfinite(x) & (x > 0), x, np.nan)
    with np.errstate(all="ignore"):
        return np.exp(np.nanmean(np.log(x), axis))


def first_above(m, thr):
    """m indexed by step n = 1..S (array index n-1). First n with m > thr; S+1 if never (censored)."""
    idx = np.where(np.asarray(m) > thr)[0]
    return int(idx[0] + 1) if idx.size else len(m) + 1


# ------------------------------------------------------------------ Q2: the Jacobian one wave at a time
def dft(N):
    x = np.arange(N)
    return np.exp(-2j * np.pi * np.outer(x, x) / N) / np.sqrt(N)


def complex_parts(J, N):
    """Real Jacobian of a 2-channel field (u0, u1) as a map on A = u0 + i u1:  dA' = alpha dA + beta conj(dA)."""
    J00, J01, J10, J11 = J[:N, :N], J[:N, N:], J[N:, :N], J[N:, N:]
    return 0.5 * ((J00 + J11) + 1j * (J10 - J01)), 0.5 * ((J00 - J11) + 1j * (J10 + J01))


def mode_stats(J, nch, N, F=None):
    """What one step does to a single Fourier mode, for each wavenumber.
    Returns k (wavenumber index), G (complex gain on the mode itself), leak (share of the output energy
    that goes to OTHER modes).  nch = 1: real field, k = 0..N/2.  nch = 2: complex field A = u0 + i u1,
    k = -N/2..N/2-1 (fftfreq order), and the conj(dA) part of the map is included in the output."""
    F = dft(N) if F is None else F
    if nch == 1:
        Jh = F @ J @ F.conj().T
        ks = np.arange(N // 2 + 1)
        G = Jh[ks, ks]; col = (np.abs(Jh[:, ks]) ** 2).sum(0)
        return ks, G, np.where(col > 0, 1 - np.abs(G) ** 2 / np.where(col > 0, col, 1), 0.0)
    if nch != 2:
        raise ValueError(nch)
    a, b = complex_parts(J, N)
    Ah, Bh = F @ a @ F.conj().T, F @ b @ F.conj().T
    idx = np.arange(N)
    out = Ah[:, idx] + Bh[:, (-idx) % N]                  # response to the input mode e^{ikx} in A
    G = out[idx, idx]; col = (np.abs(out) ** 2).sum(0)
    leak = np.where(col > 0, 1 - np.abs(G) ** 2 / np.where(col > 0, col, 1), 0.0)
    k = np.fft.fftfreq(N, 1.0 / N).astype(int)
    return k, G, leak


def fold_abs_k(k, val, N, how="median"):
    """Collapse per-k values onto |k| = 0..N/2 (for the complex field +k and -k are separate modes)."""
    out = np.full(N // 2 + 1, np.nan)
    for a in range(N // 2 + 1):
        sel = np.abs(k) == a
        if sel.any():
            out[a] = np.median(val[sel]) if how == "median" else np.sum(val[sel])
    return out


def spectrum(x, nch, N):
    """Energy per |k| = 0..N/2 of real flattened vectors x (..., nch*N); sums to |x|^2."""
    x = np.asarray(x)
    if nch == 1:
        X = np.abs(np.fft.rfft(x, axis=-1)) ** 2 / N
        X[..., 1:(N + 1) // 2] *= 2
        return X
    A = x[..., :N] + 1j * x[..., N:2 * N]
    E = np.abs(np.fft.fft(A, axis=-1)) ** 2 / N
    k = np.abs(np.fft.fftfreq(N, 1.0 / N).astype(int))
    out = np.zeros(x.shape[:-1] + (N // 2 + 1,))
    for a in range(N // 2 + 1):
        out[..., a] = E[..., k == a].sum(-1)
    return out


def mean_basis(nch, N):
    """Orthonormal basis (D, nch) of the mean (k = 0) of each channel."""
    B = np.zeros((nch * N, nch))
    for c in range(nch):
        B[c * N:(c + 1) * N, c] = 1 / np.sqrt(N)
    return B


def lyapunov(J, k, burn=0, rng=None):
    """Finite-time Lyapunov exponents (per step) of the product J[T-1] ... J[0], top k, by repeated QR.
    The first `burn` steps only align the frame and are not averaged."""
    rng = np.random.default_rng(0) if rng is None else rng
    Q = np.linalg.qr(rng.normal(size=(J.shape[-1], k)))[0]
    acc = np.zeros(k)
    for n in range(J.shape[0]):
        Q, R = np.linalg.qr(J[n] @ Q)
        s = np.sign(np.diag(R)); s[s == 0] = 1
        Q = Q * s
        if n >= burn: acc += np.log(np.abs(np.diag(R)) + 1e-300)
    return acc / max(J.shape[0] - burn, 1)


def overlap(V1, V2):
    """Overlap of two k-dimensional subspaces (orthonormal columns): 1 = same, about k/D = unrelated."""
    return float(np.linalg.norm(V1.T @ V2) ** 2 / V1.shape[1])


# ------------------------------------------------------------------ Q3: propagators, growth directions, counterfactual
def propagators(J, n):
    """[Phi(j+1 -> n) for j = 0..n-1].  Needs J[0..n-1]."""
    D = J.shape[-1]; out = [None] * n; M = np.eye(D)
    for j in range(n - 1, -1, -1):
        out[j] = M.copy(); M = M @ J[j]
    return out


def fixed_horizon(J, H, jmax):
    """[Phi(j+1 -> j+1+H) = J[j+H] ... J[j+1] for j = 0..jmax-1].  Needs J[1..jmax-1+H]."""
    D = J.shape[-1]; out = []
    for j in range(jmax):
        M = np.eye(D)
        for m in range(j + 1, j + 1 + H):
            M = J[m] @ M
        out.append(M)
    return out


def top_right(Phi, k):
    """Top-k right singular vectors of Phi (the directions it stretches most), via eigh(Phi^T Phi). (D, k)."""
    w, V = np.linalg.eigh(Phi.T @ Phi)
    return V[:, ::-1][:, :k]


def counterfactual(Phi_list, d, V_list):
    """Remove from each d_j its part in span(V_list[j]); propagate both versions with Phi_list.
    Returns error share removed (norm ratio), mistake share removed as energy and as norm, |e_n| before/after."""
    D = d.shape[1]; e = np.zeros(D); ec = np.zeros(D); dd = dr = 0.0
    for j, (Phi, V) in enumerate(zip(Phi_list, V_list)):
        rem = V @ (V.T @ d[j]) if V.shape[1] else np.zeros(D)
        e += Phi @ d[j]; ec += Phi @ (d[j] - rem)
        dd += d[j] @ d[j]; dr += rem @ rem
    ne, nc = np.linalg.norm(e), np.linalg.norm(ec)
    return dict(err=1 - nc / ne, energy=dr / dd, norm=1 - np.sqrt(max(1 - dr / dd, 0.0)), e_before=ne, e_after=nc)


def counterfactual_stack(Phi, d, V):
    """Vectorised counterfactual: Phi (n, D, D), d (n, D), V (n, D, k) or (D, k) shared. Same output as counterfactual()."""
    V = np.broadcast_to(V, (d.shape[0],) + V.shape[-2:]) if V.ndim == 2 else V
    rem = np.einsum("jdk,jk->jd", V, np.einsum("jdk,jd->jk", V, d))
    e = np.einsum("jab,jb->a", Phi, d); ec = np.einsum("jab,jb->a", Phi, d - rem)
    dd, dr = float((d * d).sum()), float((rem * rem).sum())
    ne, nc = np.linalg.norm(e), np.linalg.norm(ec)
    return dict(err=1 - nc / ne, energy=dr / dd, norm=1 - np.sqrt(max(1 - dr / dd, 0.0)), e_before=ne, e_after=nc)


def random_dirs(D, k, rng):
    return np.linalg.qr(rng.normal(size=(D, k)))[0]


def randomized_top_right(Aop, ATop, D, k, p=8, q=1, rng=None, xp=np, Omega=None):
    """Top-k right singular vectors of a linear map known only through products.
    Aop(X) = A X and ATop(Y) = A^T Y for (D, m) blocks.  Halko-Martinsson-Tropp with q power iterations.
    Works with xp = numpy or jax.numpy (pass Omega for jax)."""
    if Omega is None:
        Omega = (np.random.default_rng(0) if rng is None else rng).normal(size=(D, k + p))
    Y = Aop(Omega)
    for _ in range(q):
        Y = xp.linalg.qr(Y)[0]
        Y = Aop(ATop(Y))
    Q = xp.linalg.qr(Y)[0]
    B = ATop(Q).T                      # B = Q^T A, shape (k+p, D)
    _, _, Vt = xp.linalg.svd(B, full_matrices=False)
    return Vt[:k].T


def targets(nstar, fracs=(0.25, 0.5, 0.75), nmin=4, nmax=100):
    """Q3 horizons as fractions of the model's own validity window, clipped; empty if the window is too short."""
    if nstar < 2 * nmin:
        return []
    return sorted({int(min(max(round(f * nstar), nmin), nmax)) for f in fracs})


# ------------------------------------------------------------------ known-answer tests
def self_test(verbose=True):
    rng = np.random.default_rng(0); ok = True
    def rep(name, val, tol):
        nonlocal ok
        good = bool(np.isfinite(val) and val < tol); ok &= good
        if verbose: print(f"[{'PASS' if good else 'FAIL'}] ccore: {name}: {val:.2e} (tolerance {tol:g})")
    D, T = 12, 25
    J = rng.normal(size=(T, D, D)) / np.sqrt(D) * 1.05; d = rng.normal(size=(T, D))
    e = predict(np.repeat(np.eye(D)[None], T, 0), np.repeat(d[:1], T, 0))
    rep("ledger with J = I gives e_n = n d", np.abs(e[T - 1] - T * d[0]).max(), 1e-12)
    e = predict(J, d); n = 17; Ph = propagators(J, n)
    rep("ledger = sum_j Phi(j+1->n) d_j", np.linalg.norm(e[n - 1] - sum(Ph[j] @ d[j] for j in range(n))) / np.linalg.norm(e[n - 1]), 1e-12)
    a, b = rng.normal(size=(50, D)), rng.normal(size=(50, D)); s = split(a, b)
    rep("split: T^2 = M^2 + r P^2", np.abs(s["T"] ** 2 - s["M"] ** 2 - s["r"] * s["P"] ** 2).max(), 1e-12)
    s = split(3 * b, b); rep("split: parallel prediction has P = 0, M = 2", abs(s["P"]).max() + abs(s["M"] - 2).max(), 1e-7)
    rep("first_above: censored = S + 1", abs(first_above(np.zeros(10), 0.25) - 11), 0.5)
    rep("first_above: index is the step", abs(first_above(np.array([0, 0, .3, .5]), 0.25) - 3), 0.5)
    rep("gmean ignores non-finite", abs(gmean(np.array([[1.0], [4.0], [np.inf]]))[0] - 2), 1e-12)
    r0 = counterfactual(Ph, d[:n], [np.zeros((D, 0))] * n); rD = counterfactual(Ph, d[:n], [np.eye(D)] * n)
    rep("counterfactual: k = 0 removes nothing", abs(r0["err"]) + abs(r0["energy"]), 1e-12)
    rep("counterfactual: k = D removes everything", abs(1 - rD["err"]) + abs(1 - rD["energy"]) + abs(1 - rD["norm"]), 1e-12)
    rep("top direction of diag(1, 5, 2) is e_2", 1 - abs(top_right(np.diag([1.0, 5.0, 2.0]), 1)[1, 0]), 1e-12)
    A = rng.normal(size=(9, 9)); V = top_right(A, 3); Vs = np.linalg.svd(A)[2][:3].T
    rep("top_right spans the SVD's top-3 right vectors", 1 - overlap(V, Vs), 1e-10)
    U0, _ = np.linalg.qr(rng.normal(size=(60, 60))); W0, _ = np.linalg.qr(rng.normal(size=(60, 60)))
    A = U0 @ np.diag(0.6 ** np.arange(60)) @ W0.T
    Vr = randomized_top_right(lambda X: A @ X, lambda Y: A.T @ Y, 60, 4, rng=rng)
    rep("randomized SVD recovers the top-4 subspace", 1 - overlap(Vr, np.linalg.svd(A)[2][:4].T), 1e-8)
    Vs = [random_dirs(D, 3, rng) for _ in range(n)]
    r1, r2 = counterfactual(Ph, d[:n], Vs), counterfactual_stack(np.stack(Ph), d[:n], np.stack(Vs))
    rep("vectorised counterfactual = loop version", max(abs(r1[q] - r2[q]) for q in ("err", "energy", "norm")), 1e-12)
    Jfh = fixed_horizon(J, 3, 5)
    rep("fixed horizon = J[j+3] J[j+2] J[j+1]", np.abs(Jfh[2] - J[5] @ J[4] @ J[3]).max(), 1e-12)
    lam = lyapunov(np.repeat(np.diag([2.0, 0.5, 1.5, 0.1])[None], 200, 0), 2, burn=80)
    rep("Lyapunov exponents of diag(2, .5, 1.5, .1)", np.abs(lam - np.log([2.0, 1.5])).max(), 1e-10)
    N = 16; S = np.roll(np.eye(N), 1, axis=0)
    k, G, lk = mode_stats(S, 1, N)
    rep("1 channel, shift: gain exp(-2 pi i k/N), no leakage", np.abs(G - np.exp(-2j * np.pi * k / N)).max() + np.abs(lk).max(), 1e-12)
    J2 = np.block([[S, 0 * S], [0 * S, S]])
    k, G, lk = mode_stats(J2, 2, N)
    rep("2 channels, shift on both: complex gain exp(-2 pi i k/N), no leakage", np.abs(G - np.exp(-2j * np.pi * k / N)).max() + np.abs(lk).max(), 1e-12)
    J2c = np.block([[np.eye(N), 0 * S], [0 * S, -np.eye(N)]])        # dA' = conj(dA): pure antilinear
    k, G, lk = mode_stats(J2c, 2, N)
    rep("2 channels, conjugation: all energy leaves the mode (k != 0, N/2)", np.abs(lk[(k != 0) & (np.abs(k) != N // 2)] - 1).max(), 1e-12)
    th = 0.7; J2r = np.block([[np.cos(th) * np.eye(N), -np.sin(th) * np.eye(N)], [np.sin(th) * np.eye(N), np.cos(th) * np.eye(N)]])
    k, G, lk = mode_stats(J2r, 2, N)
    rep("2 channels, rotation by theta: gain exp(i theta)", np.abs(G - np.exp(1j * th)).max(), 1e-12)
    x = rng.normal(size=(3, N)); rep("spectrum sums to |x|^2 (1 channel)", np.abs(spectrum(x, 1, N).sum(-1) - (x ** 2).sum(-1)).max(), 1e-10)
    x = rng.normal(size=(3, 2 * N)); rep("spectrum sums to |x|^2 (2 channels)", np.abs(spectrum(x, 2, N).sum(-1) - (x ** 2).sum(-1)).max(), 1e-10)
    B = mean_basis(2, N); v = np.concatenate([np.full(N, 3.0), np.full(N, -1.0)])
    rep("mean basis captures a constant field exactly", np.linalg.norm(v - B @ (B.T @ v)), 1e-12)
    rep("targets: too-short window gives none", float(len(targets(5))), 0.5)
    rep("targets: fractions of the window, capped", float(abs(np.array(targets(60)) - np.array([15, 30, 45])).max() + abs(max(targets(400)) - 100)), 0.5)
    return ok


if __name__ == "__main__":
    print("ALL PASS" if self_test() else "FAILED")

In [ ]:
%%writefile pdecore.py
"""pdecore: solver, data generator and networks, copied VERBATIM from code/jacobian_study_JOB0_fno.py
(lines 56-65, 71-151, 153-207) so that checkpoints load and the test set is regenerated bit-for-bit.
Only NPTS is fixed to the production value and make_net takes the architecture explicitly."""
import jax
jax.config.update("jax_enable_x64", True)     # the stiff ETDRK4 contour needs float64
import numpy as np, jax.numpy as jnp, equinox as eqx

NPTS = 256
CFG = [
    dict(tag="ks_L60",     fam="ks",   L=60.0,  dte=0.25, nsub=1, nch=1,
         ref="Edson+2019 arXiv:1902.09651"),
    dict(tag="gks_d0.2",   fam="gks",  L=200.0, delta=0.2, nu=1.0, dte=0.25, nsub=1, nch=1,
         ref="Gomes+2015 arXiv:1502.05266"),
    dict(tag="cgle_phaseL1", fam="cgle", L=250.0, b1=3.5, b3=1.4, dte=0.2, nsub=4, nch=2,
         ref="Chate 1993 arXiv:patt-sol/9310001"),
    dict(tag="nik_r0.02",  fam="nik",  L=200.0, r=0.02, dte=10.0, nsub=5, nch=1,
         ref="Matthews&Cox 2007 arXiv:0708.3735"),
]

# ------------------------------------------- ETDRK4 in JAX, float64, differentiable
# Written in JAX (not numpy) so that jax.jvp through it gives J_P exactly.  Full-circle
# contour: the Kassam-Trefethen half circle is only valid for real L, and gKS (dispersion)
# and CGLE (complex L) both break that.
def spectral(cfg):
    L, fam, N = cfg["L"], cfg["fam"], NPTS
    if fam == "cgle":
        k = 2 * np.pi * np.fft.fftfreq(N, d=L / N)
        Lh = 1.0 - (1.0 + 1j * cfg["b1"]) * k ** 2
        b3 = cfg["b3"]
        def NL(v):
            A = jnp.fft.ifft(v)
            return jnp.fft.fft(-(b3 - 1j) * jnp.abs(A) ** 2 * A)
        return jnp.asarray(Lh, jnp.complex128), NL, True
    k = 2 * np.pi * np.fft.rfftfreq(N, d=L / N)
    if   fam == "ks":  Lh = k ** 2 - k ** 4
    elif fam == "gks": Lh = k ** 2 + 1j * cfg["delta"] * k ** 3 - cfg["nu"] * k ** 4
    elif fam == "nik": Lh = k ** 2 * (cfg["r"] - (1.0 - k ** 2) ** 2)
    else: raise ValueError(fam)
    mask = jnp.asarray((np.abs(k) < (2 / 3) * k.max()).astype(np.float64))   # 2/3 dealiasing
    kk = jnp.asarray(k)
    def NL(v):
        u = jnp.fft.irfft(v, n=N)
        return -0.5j * kk * jnp.fft.rfft(u ** 2) * mask
    return jnp.asarray(Lh, jnp.complex128), NL, False

def etdrk4_coeffs(Lh, h, M=64):
    r = np.exp(2j * np.pi * (np.arange(1, M + 1) - 0.5) / M)          # FULL circle
    LR = h * np.asarray(Lh)[:, None] + r[None, :]
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        Q  = h * np.mean((np.exp(LR / 2) - 1) / LR, 1)
        f1 = h * np.mean((-4 - LR + np.exp(LR) * (4 - 3 * LR + LR ** 2)) / LR ** 3, 1)
        f2 = h * np.mean((2 + LR + np.exp(LR) * (-2 + LR)) / LR ** 3, 1)
        f3 = h * np.mean((-4 - 3 * LR - LR ** 2 + np.exp(LR) * (4 - LR)) / LR ** 3, 1)
    out = [np.exp(h * np.asarray(Lh)), np.exp(h * np.asarray(Lh) / 2), Q, f1, f2, f3]
    return [jnp.asarray(np.where(np.isfinite(a), a, 0.0), jnp.complex128) for a in out]

def make_stepper(cfg):
    """Returns step_real: (nch, N) float64 -> (nch, N) float64, one emulator step."""
    Lh, NL, cx = spectral(cfg)
    h = cfg["dte"] / cfg["nsub"]
    E, E2, Q, f1, f2, f3 = etdrk4_coeffs(Lh, h)
    N = NPTS
    def to_spec(u):   # (nch,N) real -> spectral
        return jnp.fft.fft(u[0] + 1j * u[1]) if cx else jnp.fft.rfft(u[0])
    def to_real(v):
        if cx:
            A = jnp.fft.ifft(v); return jnp.stack([A.real, A.imag], 0)
        return jnp.fft.irfft(v, n=N)[None]
    def sub(v):
        Nv = NL(v);  a = E2 * v + Q * Nv
        Na = NL(a);  b = E2 * v + Q * Na
        Nb = NL(b);  c = E2 * a + Q * (2 * Nb - Nv)
        Nc = NL(c)
        return E * v + Nv * f1 + 2 * (Na + Nb) * f2 + Nc * f3
    def step_real(u):
        v = to_spec(u)
        v = jax.lax.fori_loop(0, cfg["nsub"], lambda _, z: sub(z), v)
        return to_real(v)
    return jax.jit(step_real)

def random_ic(cfg, rng, n, nm=6):
    x = np.linspace(0, cfg["L"], NPTS, endpoint=False)
    u = np.zeros((n, cfg["nch"], NPTS))
    for i in range(n):
        for ch in range(cfg["nch"]):
            for m in range(1, nm + 1):
                u[i, ch] += rng.normal() / m * np.sin(2*np.pi*m*x/cfg["L"] + rng.uniform(0, 2*np.pi))
    return 0.5 * u / (np.abs(u).max(axis=(1, 2), keepdims=True) + 1e-9)

def make_data(cfg, step, ntraj, nsteps, seed, warm):
    rng = np.random.default_rng(seed)
    u = jnp.asarray(random_ic(cfg, rng, ntraj), jnp.float64)
    roll = jax.jit(lambda u0, m: jax.lax.scan(lambda s, _: (jax.vmap(step)(s),) * 2, u0, None, length=m),
               static_argnums=1)
    u, _ = roll(u, warm)
    _, tr = roll(u, nsteps)                                   # (nsteps, ntraj, nch, N)
    out = jnp.concatenate([u[None], tr], 0).swapaxes(0, 1)    # (ntraj, nsteps+1, nch, N)
    if not bool(jnp.isfinite(out).all()) or float(jnp.abs(out).max()) > 1e3:
        raise RuntimeError(f"{cfg['tag']}: solver unstable (max {float(jnp.abs(out).max()):.3g})")
    return np.asarray(out, np.float32)

# ------------------------------------------------------------------------ networks
class SpectralConv1d(eqx.Module):
    wr: jnp.ndarray; wi: jnp.ndarray; modes: int = eqx.field(static=True)
    def __init__(self, cin, cout, modes, key):
        sc = 1.0 / (cin * cout) ** 0.5; k1, k2 = jax.random.split(key)
        self.wr = jax.random.normal(k1, (cin, cout, modes), dtype=jnp.float32) * sc
        self.wi = jax.random.normal(k2, (cin, cout, modes), dtype=jnp.float32) * sc
        self.modes = modes
    def __call__(self, x):
        xh = jnp.fft.rfft(x, axis=-1)
        m = min(self.modes, xh.shape[-1])
        w = (self.wr + 1j * self.wi)[:, :, :m]
        o = jnp.einsum("in,ion->on", xh[:, :m], w)
        full = jnp.zeros((w.shape[1], xh.shape[-1]), dtype=o.dtype).at[:, :m].set(o)
        return jnp.fft.irfft(full, n=x.shape[-1], axis=-1)

class Conv1(eqx.Module):
    w: jnp.ndarray; b: jnp.ndarray; k: int = eqx.field(static=True)
    def __init__(self, cin, cout, k, key, zero=False):
        self.w = (jnp.zeros((cout, cin, k), jnp.float32) if zero else
                  jax.random.normal(key, (cout, cin, k), dtype=jnp.float32) / (cin * k) ** 0.5)
        self.b = jnp.zeros(cout, jnp.float32); self.k = k
    def __call__(self, x):
        p = self.k // 2
        xp = jnp.concatenate([x[:, -p:], x, x[:, :p]], -1)[None] if p else x[None]
        y = jax.lax.conv_general_dilated(xp, self.w, (1,), "VALID",
                                         dimension_numbers=("NCW", "OIW", "NCW"))[0]
        return y + self.b[:, None]

class FNO(eqx.Module):                       # APEBench preset FNO;12;18;4;gelu ~ 32.5k params
    lift: Conv1; spec: list; pw: list; p1: Conv1; p2: Conv1
    def __init__(self, nch, key, width=18, modes=12, nblk=4):
        ks = jax.random.split(key, 2 * nblk + 3)
        self.lift = Conv1(nch, width, 1, ks[0])
        self.spec = [SpectralConv1d(width, width, modes, ks[1 + i]) for i in range(nblk)]
        self.pw   = [Conv1(width, width, 1, ks[1 + nblk + i]) for i in range(nblk)]
        self.p1 = Conv1(width, 64, 1, ks[-2]); self.p2 = Conv1(64, nch, 1, ks[-1], zero=True)
    def __call__(self, u):
        x = self.lift(u)
        for sc, pc in zip(self.spec, self.pw): x = jax.nn.gelu(sc(x) + pc(x))
        return u + self.p2(jax.nn.gelu(self.p1(x)))

class ResNet(eqx.Module):                    # APEBench preset Res;26;8;relu ~ 32.9k params
    lift: Conv1; blk: list; proj: Conv1
    def __init__(self, nch, key, width=26, nblk=8):
        ks = jax.random.split(key, 2 * nblk + 2)
        self.lift = Conv1(nch, width, 3, ks[0])
        self.blk = [(Conv1(width, width, 3, ks[1 + 2*i]), Conv1(width, width, 3, ks[2 + 2*i]))
                    for i in range(nblk)]
        self.proj = Conv1(width, nch, 3, ks[-1], zero=True)
    def __call__(self, u):
        x = self.lift(u)
        for a, b in self.blk: x = x + b(jax.nn.relu(a(x)))
        return u + self.proj(x)


def make_net(arch, nch, key):
    return FNO(nch, key) if arch == "fno" else ResNet(nch, key)

In [ ]:
%%writefile oagg.py
"""oagg: tables and figures for the own-error Jacobian runs (numpy + matplotlib only).
Results: <RES>/own__<mode>__<pde>__<arch>__s<seed>__<arm>__<ck>.npz   arm = base (the dataset's one-step model), mse, randR,
ownR, owncosR, bothR, cosC, randcosR_C;  ck = best / final.   History: <HIS>/own__<mode>__<pde>__<arch>__s<seed>__<arm>.json"""
import os, glob, json, csv, re
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

STEPS = (50, 100, 200)
KIND_ORDER = {"base": 0, "mse": 1, "rand": 2, "own": 3, "owncos": 4, "both": 5, "cos": 6, "randcos": 7}
KIND_COLOR = {"base": "#0b0b0b", "mse": "#2a78d6", "rand": "#eb6834", "owncos": "#e87ba4", "both": "#6250d6", "cos": "#eda100"}
OWN_SHADES = ["#8fdcbd", "#1baf7a", "#0d6b4a", "#063d2a"]
RC_SHADES = ["#f2a3a2", "#e34948", "#a51f1e", "#5e0e0d"]


def gmean(x, axis=0):
    x = np.where(np.isfinite(x) & (x > 0), x, np.nan)
    with np.errstate(all="ignore"):
        return np.exp(np.nanmean(np.log(x), axis))


def first_above(m, thr):
    i = np.where(np.asarray(m) > thr)[0]
    return int(i[0] + 1) if i.size else len(m) + 1


def parse(arm):
    """-> (kind, sort value); for randcosR_C the sort value is C (the cosine ratio)."""
    if arm in ("base", "mse"):
        return arm, 0.0
    m = re.fullmatch(r"randcos([0-9p.]+)_([0-9p.]+)", arm)
    if m:
        return "randcos", float(m.group(2).replace("p", "."))
    m = re.fullmatch(r"(owncos|own|rand|both|cos)([0-9p.]+)", arm)
    return (m.group(1), float(m.group(2).replace("p", "."))) if m else (arm, 0.0)


def arm_key(arm):
    k, r = parse(arm)
    return (KIND_ORDER.get(k, 9), r)


def arm_color(arm, arms):
    k, r = parse(arm)
    if k in ("own", "randcos"):
        sh = OWN_SHADES if k == "own" else RC_SHADES
        rs = sorted({parse(a)[1] for a in arms if parse(a)[0] == k})
        return sh[min(rs.index(r), len(sh) - 1)] if r in rs else sh[1]
    return KIND_COLOR.get(k, "#52514e")


def arm_label(arm):
    k, r = parse(arm)
    rr = f"{r:g}"
    if k == "randcos":
        m = re.fullmatch(r"randcos([0-9p.]+)_([0-9p.]+)", arm)
        return f"+ random gap (x{m.group(1)}) + own-error cosine (x{m.group(2)})"
    return {"base": "dataset one-step model", "mse": "one-step loss (control)", "rand": f"+ random-probe gap (x{rr})",
            "own": f"+ own-error gap (x{rr})", "owncos": f"+ own-error size/direction (x{rr})",
            "both": f"+ random + own-error gaps (x{rr})", "cos": f"+ own-error cosine (x{rr})"}.get(k, arm)


def load(res):
    out = []
    for p in sorted(glob.glob(os.path.join(res, "own__*.npz"))):
        b = os.path.basename(p)[:-4].split("__")
        if len(b) != 7:
            continue
        with np.load(p, allow_pickle=False) as z:
            out.append(dict(mode=b[1], pde=b[2], arch=b[3], seed=int(b[4][1:]), arm=b[5], ck=b[6], z={k: z[k] for k in z.files}))
    return out


def load_hist(his):
    H = {}
    for p in glob.glob(os.path.join(his, "own__*.json")):
        b = os.path.basename(p)[:-5].split("__")
        try:
            H[(b[1], b[2], b[3], int(b[4][1:]), b[5])] = json.load(open(p))
        except (OSError, ValueError, IndexError):
            pass
    return H


def metrics(z):
    S = z["rel_true"].shape[1]; r = {}
    rt = gmean(z["rel_true"])
    with np.errstate(all="ignore"):
        ratio = z["rel_true"] / z["rel_P"]
    for n in STEPS:
        if n <= S:
            r[f"err_{n}"] = float(rt[n - 1]); r[f"true_over_eP_{n}"] = float(gmean(ratio[:, n - 1]))
    r["err_end"] = float(rt[S - 1])
    r["onestep"] = float(gmean(z["d_rel"].reshape(-1)))
    for k in ("own_gap", "own_gap_rel", "rand_gap", "rand_gap_rel", "own_1mcos"):
        if k in z:
            r[k] = float(z[k])
    lam = float(z["lam"])
    r["vpt_lt"] = float(np.median(z["vpt_steps"])) * lam
    for th in (90, 80):
        if f"tcorr{th}_steps" in z:
            r[f"tcorr{th}_lt"] = float(np.median(z[f"tcorr{th}_steps"])) * lam
    r["Nf25"] = first_above(gmean(z["T_f"]), 0.25); r["NP25"] = first_above(gmean(z["T_P"]), 0.25)
    r["n_div"] = int((z["div_step"] <= S).sum())
    if "lyap_f" in z:
        r["lyap1_f"] = float(np.nanmedian(z["lyap_f"][:, 0])); r["lyap1_P"] = float(np.nanmedian(z["lyap_P"][:, 0]))
    return r


RATIO_COLS = [f"err_{n}" for n in STEPS] + ["err_end", "onestep", "own_gap", "rand_gap", "own_1mcos", "vpt_lt", "tcorr80_lt"]


def write_csv(path, rows):
    if not rows:
        return
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(cols)
        for r in rows:
            w.writerow(["" if isinstance(r.get(c), float) and not np.isfinite(r.get(c)) else
                        (f"{r[c]:.4g}" if isinstance(r.get(c), float) else r.get(c, "")) for c in cols])


def _style(ax):
    ax.grid(True, which="both", lw=0.4, color="#e4e3df"); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def aggregate(res, his, out, verbose=True):
    recs = load(res); H = load_hist(his)
    M = {(x["mode"], x["pde"], x["arch"], x["seed"], x["arm"], x["ck"]): metrics(x["z"]) for x in recs}
    rows = []
    for key, m in M.items():
        mode, pde, arch, seed, arm, ck = key
        row = dict(mode=mode, pde=pde, arch=arch, seed=seed, arm=arm, ck=ck, **m)
        c = M.get((mode, pde, arch, seed, "mse", ck))
        b = M.get((mode, pde, arch, seed, "base", "best"))
        for col in RATIO_COLS:
            if c is not None and col in m and col in c and arm not in ("mse", "base"):
                row[f"{col}_vs_ctrl"] = m[col] / c[col] if c[col] else float("nan")
            if b is not None and col in m and col in b and arm != "base":
                row[f"{col}_vs_data"] = m[col] / b[col] if b[col] else float("nan")
        h = H.get((mode, pde, arch, seed, arm))
        if h:
            row.update(lam_own=h.get("lam_own", 0.0), lam_rand=h.get("lam_rand", 0.0), lam_cos=h.get("lam_cos", 0.0), best_it=h.get("best_it"),
                       updates=h.get("updates"), minutes=h.get("minutes"), stopped=h.get("stopped", ""))
        rows.append(row)
    rows.sort(key=lambda r: (r["mode"], r["pde"], r["arch"], r["ck"], arm_key(r["arm"]), r["seed"]))
    write_csv(os.path.join(out, "table_own_runs.csv"), rows)
    summ = []
    keys = sorted({(r["mode"], r["pde"], r["arch"], r["arm"], r["ck"]) for r in rows}, key=lambda k: (k[0], k[1], k[2], k[4], arm_key(k[3])))
    for key in keys:
        sel = [r for r in rows if (r["mode"], r["pde"], r["arch"], r["arm"], r["ck"]) == key]
        s = dict(mode=key[0], pde=key[1], arch=key[2], arm=key[3], ck=key[4], n_seeds=len(sel))
        for col in sel[0]:
            if col in s or col in ("seed", "stopped"):
                continue
            v = np.array([q.get(col, np.nan) if isinstance(q.get(col, np.nan), (int, float)) and q.get(col) is not None else np.nan for q in sel], float)
            v = v[np.isfinite(v)]
            s[col] = float(np.median(v)) if v.size else float("nan")
        if key[3] not in ("mse", "base") and all("err_end_vs_ctrl" in q for q in sel):
            b100 = sum(q.get("err_100_vs_ctrl", q["err_end_vs_ctrl"]) < 1 for q in sel); b200 = sum(q["err_end_vs_ctrl"] < 1 for q in sel)
            dok = all(q["onestep_vs_ctrl"] <= 1.0 for q in sel)
            s["seeds_better_100"] = f"{b100}/{len(sel)}"; s["seeds_better_200"] = f"{b200}/{len(sel)}"
            s["success"] = "YES" if (b100 == len(sel) and b200 == len(sel) and dok) else "no"
        s["n_div"] = int(sum(q.get("n_div", 0) for q in sel))
        summ.append(s)
    write_csv(os.path.join(out, "table_own_summary.csv"), summ)
    for mode in sorted({r["mode"] for r in rows}):
        figures(recs, rows, H, out, mode)
    if verbose:
        show = [("arm", 14), ("err_100_vs_ctrl", 9), ("err_200_vs_ctrl", 9), ("onestep_vs_ctrl", 9), ("own_gap_vs_ctrl", 9), ("own_1mcos_vs_ctrl", 9),
                ("rand_gap_vs_ctrl", 9), ("vpt_lt", 7), ("err_200_vs_data", 9), ("seeds_better_200", 7), ("success", 7)]
        for mode in sorted({s["mode"] for s in summ}):
            for pde in sorted({s["pde"] for s in summ if s["mode"] == mode}):
                print(f"\n{mode} | {pde} (best checkpoints, median over seeds; *_vs_ctrl: ratio to the one-step-loss control)")
                print("  " + " ".join(f"{c[:w]:>{w}}" for c, w in show))
                for s in summ:
                    if s["mode"] == mode and s["pde"] == pde and s["ck"] == "best":
                        print("  " + " ".join(f"{(f'{s[c]:.3g}' if isinstance(s.get(c), float) else str(s.get(c, '')))[:w]:>{w}}" for c, w in show))
    return rows, summ


def figures(recs, rows, H, out, mode):
    pdes = sorted({x["pde"] for x in recs if x["mode"] == mode})
    if not pdes:
        return
    arms = sorted({x["arm"] for x in recs if x["mode"] == mode}, key=arm_key)
    # 1. error curves and per-seed ratio to the control
    fig, axs = plt.subplots(len(pdes), 2, figsize=(11.5, 3.4 * len(pdes)), dpi=130, squeeze=False,
                            gridspec_kw=dict(width_ratios=[1.35, 1]))
    for i, pde in enumerate(pdes):
        a, b = axs[i, 0], axs[i, 1]; _style(a); _style(b)
        for arm in arms:
            sel = [x["z"] for x in recs if x["mode"] == mode and x["pde"] == pde and x["arm"] == arm and x["ck"] == "best"]
            if not sel:
                continue
            rt = np.concatenate([z["rel_true"] for z in sel]); n = np.arange(1, rt.shape[1] + 1)
            a.plot(n, gmean(rt), color=arm_color(arm, arms), lw=2.2 if arm in ("base", "mse") else 1.4,
                   ls="--" if arm == "base" else "-", label=arm_label(arm))
        a.set(yscale="log", ylabel="relative error (geo. mean)", title=f"{pde}: rollout error, best checkpoints, all seeds")
        if i == len(pdes) - 1:
            a.set_xlabel("rollout step n")
        if i == 0:
            a.legend(fontsize=6.8, frameon=False)
        cand = [arm for arm in arms if arm not in ("base", "mse")]
        for j, arm in enumerate(cand):
            sel = [r for r in rows if r["mode"] == mode and r["pde"] == pde and r["arm"] == arm and r["ck"] == "best"]
            for r in sel:
                for col, mk, off in (("err_100_vs_ctrl", "o", -0.12), ("err_end_vs_ctrl", "s", 0.12)):
                    if col in r and np.isfinite(r[col]):
                        b.scatter(j + off, r[col], color=arm_color(arm, arms), marker=mk, s=28,
                                  facecolors="none" if col == "err_100_vs_ctrl" else arm_color(arm, arms), zorder=3)
        b.axhline(1, color="#0b0b0b", lw=0.9)
        b.set_xticks(range(len(cand)), cand, rotation=30, ha="right", fontsize=8)
        b.set(yscale="log", ylabel="error / control's (same seed)", title="per seed: step 100 (open), last step (filled)")
    fig.suptitle(f"One-step loss + Jacobian gap terms on the ResNets ({mode}); below 1 = better than the one-step-loss control", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(out, f"fig_own_error_{mode}.png"), bbox_inches="tight"); plt.close(fig)
    # 2. trade-off relative to the control
    fig, axs = plt.subplots(1, len(pdes), figsize=(4.4 * len(pdes), 3.8), dpi=130, squeeze=False)
    for j, pde in enumerate(pdes):
        ax = axs[0, j]; _style(ax)
        for arm in arms:
            sel = [r for r in rows if r["mode"] == mode and r["pde"] == pde and r["arm"] == arm and r["ck"] == "best"
                   and "onestep_vs_ctrl" in r and "err_end_vs_ctrl" in r]
            if sel:
                ax.scatter([r["onestep_vs_ctrl"] for r in sel], [r["err_end_vs_ctrl"] for r in sel], color=arm_color(arm, arms),
                           s=30, label=arm_label(arm), zorder=3)
        ax.axhline(1, color="#0b0b0b", lw=0.8); ax.axvline(1, color="#0b0b0b", lw=0.8)
        ax.set(xscale="log", yscale="log", xlabel="one-step mistake / control's", ylabel="error at the last step / control's", title=pde)
        if j == 0:
            ax.legend(fontsize=6.5, frameon=False)
    fig.suptitle(f"Per seed, relative to the control ({mode}); lower-left quadrant = better on both", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(out, f"fig_own_tradeoff_{mode}.png"), bbox_inches="tight"); plt.close(fig)
    # 3. training curves
    hs = {k: v for k, v in H.items() if k[0] == mode}
    if not hs:
        return
    qs = [("val_rollout", "val rollout error (100 steps)"), ("val_onestep_rel", "val one-step error"),
          ("val_own_gap_rel", "val gap along own error / |J_P e|"), ("val_own_1mcos", "val 1 - cos(J_f e, J_P e)"),
          ("val_rand_gap_rel", "val random-direction gap / |J_P v|")]
    fig, axs = plt.subplots(len(pdes), len(qs), figsize=(4.0 * len(qs), 2.9 * len(pdes)), dpi=120, squeeze=False)
    for i, pde in enumerate(pdes):
        for j, (q, lab) in enumerate(qs):
            ax = axs[i, j]; _style(ax)
            for (md_, p_, a_, s_, arm), h in sorted(hs.items(), key=lambda kv: arm_key(kv[0][4])):
                if p_ != pde:
                    continue
                hh = h.get("hist", [])
                ax.plot([x["it"] for x in hh], [x.get(q, np.nan) for x in hh], color=arm_color(arm, arms), lw=1.1,
                        alpha=1.0 if s_ == min(k[3] for k in hs) else 0.45, label=arm if s_ == min(k[3] for k in hs) else None)
            ax.set(yscale="log", ylabel=lab, title=pde if j == 0 else "")
            if i == len(pdes) - 1:
                ax.set_xlabel("update")
            if i == 0 and j == 0:
                ax.legend(fontsize=6.5, frameon=False)
    fig.tight_layout(); fig.savefig(os.path.join(out, f"fig_own_training_{mode}.png"), bbox_inches="tight"); plt.close(fig)

In [ ]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp, equinox as eqx, optax
import pdecore as pc, ccore as cc, oagg
assert jnp.zeros(1).dtype == jnp.float64
print("jax", jax.__version__, jax.devices(), "| MODE", MODE, "| SMOKE", SMOKE, "| PDES", PDES, "| ARMS", ARMS)

OUT  = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("OUT", "./own_out")
BASE = os.path.join(OUT, "own")
RES, CKD, HIS = (os.path.join(BASE, d) for d in ("res", "ckpt", "hist"))
for d in (RES, CKD, HIS): os.makedirs(d, exist_ok=True)
ROOT = os.environ.get("PDE_DATA")
if ROOT is None:
    for dp, dn, _ in os.walk("/kaggle/input"):
        if "weights_fno" in dn and "weights_resnet" in dn: ROOT = dp; break
assert ROOT, "dataset not found: attach the four-PDE emulator dataset via Add data"
WFILES = {os.path.basename(p): p for p in glob.glob(os.path.join(ROOT, "weights_*", "*.eqx"))}
RESJ = {}
for f in glob.glob(os.path.join(ROOT, "**", "job*_results.json"), recursive=True):
    RESJ.update(json.load(open(f)))
print("dataset:", ROOT, "|", len(WFILES), "checkpoints |", len(RESJ), "stored records")
n_res = 0                                          # resume, and merge other sessions' outputs if attached
for sub in ("res", "ckpt", "hist"):
    for p in glob.glob(f"/kaggle/input/**/own/{sub}/*", recursive=True):
        dst = os.path.join(BASE, sub, os.path.basename(p))
        if not os.path.exists(dst): shutil.copyfile(p, dst); n_res += 1
print("resumed / merged files:", n_res)

CFG = {c["tag"]: dict(c) for c in pc.CFG}
N = pc.NPTS
LAM = {t: float(next(r["lam"] for r in RESJ.values() if r["pde"] == t)) for t in PDES}   # Lyapunov exponent per step
STEP, TRAIN, VAL, TEST, SIG2 = {}, {}, {}, {}, {}
for t in PDES:
    t0 = time.time(); STEP[t] = pc.make_stepper(CFG[t])
    TRAIN[t] = pc.make_data(CFG[t], STEP[t], NTRAIN, TRAIN_H, 1000, WARM)
    VAL[t]   = pc.make_data(CFG[t], STEP[t], NVAL,   VAL_H,   4242, WARM)
    TEST[t]  = pc.make_data(CFG[t], STEP[t], NTEST,  S,       773,  WARM)
    SIG2[t]  = np.var(TRAIN[t].astype(np.float64), axis=(0, 1, 3))           # per-channel variance (for the VPT)
    print(f"{t:<14} train {TRAIN[t].shape} val {VAL[t].shape} test {TEST[t].shape}  lambda/step {LAM[t]:.4f}  ({time.time() - t0:.0f} s)")
OPT = optax.chain(optax.clip_by_global_norm(1.0), optax.scale_by_adam(), optax.scale(-1.0))

def template(tag): return pc.make_net(ARCH, CFG[tag]["nch"], jax.random.PRNGKey(0))
def load_dataset_model(tag, seed):
    return eqx.tree_deserialise_leaves(WFILES[f"{tag}__{ARCH}__s{seed}__best.eqx"], template(tag))
def init_model(tag, seed):                          # the dataset's own initialisation of that seed
    return pc.make_net(ARCH, CFG[tag]["nch"], jax.random.PRNGKey(seed))
def f_of(model, z): return model(z.astype(jnp.float32)).astype(jnp.float64)

def parse_arm(arm):
    """-> (kind, ratios) with ratios = dict(r=random term, o=own-error term, c=cosine term)."""
    num = lambda x: float(x.replace("p", "."))
    if arm == "mse": return "mse", dict(r=0.0, o=0.0, c=0.0)
    m = re.fullmatch(r"randcos([0-9p.]+)_([0-9p.]+)", arm)
    if m: return "randcos", dict(r=num(m.group(1)), o=0.0, c=num(m.group(2)))
    m = re.fullmatch(r"(owncos|own|rand|both|cos)([0-9p.]+)", arm)
    assert m, f"unknown arm {arm}"
    k, v = m.group(1), num(m.group(2))
    return k, dict(r=v if k in ("rand", "both") else 0.0, o=v if k in ("own", "owncos", "both") else 0.0, c=v if k == "cos" else 0.0)
for a in ARMS: parse_arm(a)
NEED_OWN = ("own", "owncos", "both", "cos", "randcos")      # kinds that probe along the model's own error
NEED_RAND = ("rand", "both", "randcos")                     # kinds that probe random directions

## The loss terms (per PDE; the model and the probe directions are arguments)

In [ ]:
def make_fns(tag):
    step, nch = STEP[tag], CFG[tag]["nch"]; D = nch * N
    f64 = jnp.float64

    def own_dirs(model, Xw):
        """Xw (B, H, nch, N) true states x_0..x_{H-1}. Rolls the model out from x_0 and returns the unit direction of its
        own error at every x_k, (B, H, nch, N) float64; k = 0 (no error yet) and non-finite errors give 0 (no probe)."""
        def roll(x0):
            return jax.lax.scan(lambda z, _: (model(z),) * 2, x0, None, length=Xw.shape[1] - 1)[1]
        Z = jax.vmap(roll)(Xw[:, 0])
        E = (Z - Xw[:, 1:]).astype(f64)
        E = jnp.where(jnp.isfinite(E), E, 0.0)
        nr = jnp.sqrt(jnp.sum(E ** 2, axis=(-2, -1), keepdims=True))
        U = jnp.where(nr > 0, E / jnp.where(nr > 0, nr, 1.0), 0.0)
        return jnp.concatenate([jnp.zeros_like(U[:, :1]), U], 1)

    def rand_dirs(key, Xw):
        V = jax.random.normal(key, Xw.shape, f64)
        V = V / jnp.sqrt(jnp.sum(V ** 2, axis=(-2, -1), keepdims=True))
        return V.at[:, 0].set(0.0)                     # same states as the own-error term: k = 1..H-1

    def gaps(model, Xw, Dirs):
        """Per probed state: |a - b|^2 / D, the size-and-direction form, |b|^2 / D, and a 0/1 mask,
        with a = J_f(x) v, b = J_P(x) v (b is data: no gradient)."""
        X = Xw.reshape((-1,) + Xw.shape[2:]); V = Dirs.reshape((-1,) + Dirs.shape[2:])
        a = jax.vmap(lambda x, v: jax.jvp(model, (x,), (v.astype(x.dtype),))[1])(X, V).astype(f64)
        b = jax.lax.stop_gradient(jax.vmap(lambda x, v: jax.jvp(step, (x.astype(f64),), (v,))[1])(X, V))
        w = (jnp.sum(V ** 2, axis=(-2, -1)) > 0.5).astype(f64)
        sab = lambda p, q: jnp.sum(p * q, axis=(-2, -1))
        l2 = sab(a - b, a - b) / D
        na, nb = jnp.sqrt(sab(a, a) + 1e-30), jnp.sqrt(sab(b, b) + 1e-30)
        cos = sab(a, b) / (na * nb)
        mc = ((na - nb) ** 2 + BETA_COS * nb ** 2 * (1.0 - cos)) / D
        return l2, mc, sab(b, b) / D, w, 1.0 - cos

    def terms(model, Xw, Yw, Down=None, Drand=None):
        d = (jax.vmap(jax.vmap(model))(Xw) - Yw).astype(f64)
        out = {"mse": jnp.mean(d ** 2)}
        wm = lambda x, w: jnp.sum(x * w) / jnp.maximum(jnp.sum(w), 1.0)
        if Down is not None:
            l2, mc, b2, w, oc = gaps(model, Xw, Down)
            out.update(own=wm(l2, w), owncos=wm(mc, w), own_b2=wm(b2, w), cos=wm(oc, w))
        if Drand is not None:
            l2, _, b2, w, _ = gaps(model, Xw, Drand)
            out.update(rand=wm(l2, w), rand_b2=wm(b2, w))
        return out

    def objective(model, Xw, Yw, key, lam_r, lam_o, lam_c, kind):
        """kind (static): mse / own / owncos / rand / both / cos / randcos. The own-error direction is computed with the current model
        and then held fixed (stop-gradient): the loss asks J_f to carry that error like J_P, not to move the error."""
        Down = jax.lax.stop_gradient(own_dirs(model, Xw)) if kind in NEED_OWN else None
        Drand = rand_dirs(key, Xw) if kind in NEED_RAND else None
        t = terms(model, Xw, Yw, Down, Drand)
        L = t["mse"]
        if kind in ("own", "both"): L = L + lam_o * t["own"]
        if kind == "owncos":        L = L + lam_o * t["owncos"]
        if kind in NEED_RAND:       L = L + lam_r * t["rand"]
        if kind in ("cos", "randcos"): L = L + lam_c * t["cos"]
        return L, t

    @eqx.filter_jit
    def probe_terms(model, Xw, Yw, key):              # all terms, for calibration and validation
        return terms(model, Xw, Yw, own_dirs(model, Xw), rand_dirs(key, Xw))

    @eqx.filter_jit
    def onestep(model, X, Y):
        d = jax.vmap(model)(X) - Y
        return jnp.mean(jnp.linalg.norm(d.reshape(d.shape[0], -1), axis=-1) / jnp.linalg.norm(Y.reshape(Y.shape[0], -1), axis=-1))
    @eqx.filter_jit
    def val_rollout(model, Vt):
        pr = jnp.swapaxes(jax.lax.scan(lambda s, _: (jax.vmap(model)(s),) * 2, Vt[:, 0], None, length=Vt.shape[1] - 1)[1], 0, 1)
        e = jnp.sqrt(jnp.sum((pr - Vt[:, 1:]) ** 2, (-2, -1)) / jnp.sum(Vt[:, 1:] ** 2, (-2, -1)))
        return jnp.exp(jnp.mean(jnp.log(jnp.clip(e.mean(0), 1e-12, 1e3))))
    @eqx.filter_jit
    def q1(model, U):
        S_ = U.shape[1] - 1
        fo = lambda z: f_of(model, z)
        pred = jnp.swapaxes(jax.lax.scan(lambda s, _: (jax.vmap(fo)(s),) * 2, U[:, 0], None, length=S_)[1], 0, 1)
        e_true = pred - U[:, 1:]
        d = jax.vmap(jax.vmap(fo))(U[:, :S_]) - jax.vmap(jax.vmap(step))(U[:, :S_])
        def prop(Jfun):
            def body(e, i):
                Je = jax.vmap(lambda b, v: jax.jvp(Jfun, (b,), (v,))[1])(U[:, i], e)
                return (Je + d[:, i],) * 2
            return jnp.swapaxes(jax.lax.scan(body, jnp.zeros_like(U[:, 0]), jnp.arange(S_))[1], 0, 1)
        return e_true, prop(fo), prop(step), d
    @eqx.filter_jit
    def gap_along(model, X, V):                       # X, V (M, nch, N) float64, V unit or 0
        a = jax.vmap(lambda x, v: jax.jvp(lambda z: f_of(model, z), (x,), (v,))[1])(X, V)
        b = jax.vmap(lambda x, v: jax.jvp(step, (x,), (v,))[1])(X, V)
        sab = lambda p, q: jnp.sum(p * q, axis=(-2, -1))
        return sab(a - b, a - b), sab(b, b), 1.0 - sab(a, b) / jnp.sqrt(sab(a, a) * sab(b, b) + 1e-300)
    @eqx.filter_jit
    def jac_f(model, X):
        return jax.vmap(jax.jacfwd(lambda z: f_of(model, z.reshape(nch, N)).reshape(-1)))(X)
    jac_P = jax.jit(jax.vmap(jax.jacfwd(lambda z: step(z.reshape(nch, N)).reshape(-1))))
    return dict(own_dirs=own_dirs, rand_dirs=rand_dirs, terms=terms, objective=objective, probe_terms=probe_terms,
                onestep=onestep, val_rollout=val_rollout, q1=q1, gap_along=gap_along, jac_f=jac_f, jac_P=jac_P, D=D)

FNS = {t: make_fns(t) for t in PDES}

def windows(arr, H, stride):
    starts = np.arange(0, arr.shape[1] - H, stride)
    X = np.stack([arr[:, s:s + H] for s in starts], 1).reshape((-1, H) + arr.shape[2:])
    Y = np.stack([arr[:, s + 1:s + H + 1] for s in starts], 1).reshape((-1, H) + arr.shape[2:])
    return jnp.asarray(X, jnp.float32), jnp.asarray(Y, jnp.float32)
VALW = {t: windows(VAL[t], HW, VAL_STRIDE) for t in PDES}
VAL1 = {t: windows(VAL[t], 1, 5) for t in PDES}
VALTRAJ = {t: jnp.asarray(VAL[t], jnp.float32) for t in PDES}
def val_terms(tag, model, ch=32):
    X, Y = VALW[tag]; acc = {}
    for i in range(0, X.shape[0], ch):
        t = FNS[tag]["probe_terms"](model, X[i:i + ch], Y[i:i + ch], jax.random.PRNGKey(99 + i))
        for k, v in t.items(): acc[k] = acc.get(k, 0.0) + float(v) * min(ch, X.shape[0] - i)
    return {k: v / X.shape[0] for k, v in acc.items()}

## Validation: every line must print PASS

In [ ]:
def check(name, val, tol):
    good = bool(np.isfinite(val) and val < tol)
    print(f"[{'PASS' if good else 'FAIL'}] {name}: {val:.2e} (tolerance {tol:g})"); return good

ok = cc.self_test(verbose=False); print(f"[{'PASS' if ok else 'FAIL'}] ccore known-answer tests")
rng = np.random.default_rng(0)
tnorm = lambda tr: float(jnp.sqrt(sum(jnp.sum(a ** 2) for a in jax.tree_util.tree_leaves(tr))))
tdot = lambda a_, b_: float(sum(jnp.sum(a * b) for a, b in zip(jax.tree_util.tree_leaves(a_), jax.tree_util.tree_leaves(b_))))
for t in PDES:
    fn, nch, D = FNS[t], CFG[t]["nch"], FNS[t]["D"]
    m = load_dataset_model(t, SEEDS[0])
    u = jnp.asarray(TRAIN[t][0, 10], jnp.float64); w = jnp.asarray(rng.normal(size=u.shape)); eps = 1e-6
    fd = (STEP[t](u + eps * w) - STEP[t](u - eps * w)) / (2 * eps)
    ok &= check(f"{t}: solver tangent vs central difference", float(jnp.linalg.norm(jax.jvp(STEP[t], (u,), (w,))[1] - fd) / jnp.linalg.norm(fd)), 1e-6)
    # every term against an explicit dense computation on one window
    H = 6; Xw = jnp.asarray(TRAIN[t][:1, 3:3 + H], jnp.float32); Yw = jnp.asarray(TRAIN[t][:1, 4:4 + H], jnp.float32)
    xs64 = jnp.asarray(TRAIN[t][0, 3:3 + H], jnp.float64).reshape(H, -1)
    JP = np.asarray(fn["jac_P"](xs64)); JF = np.asarray(fn["jac_f"](m, xs64))
    z = Xw[0, 0]; E = [np.zeros(D)]
    for k in range(1, H):
        z = m(z); E.append(np.asarray(z - Xw[0, k], np.float64).reshape(-1))
    Uref = np.stack([e / np.linalg.norm(e) if np.linalg.norm(e) > 0 else e for e in E])
    Ud = np.asarray(fn["own_dirs"](m, Xw))[0].reshape(H, -1)
    # float32 rollouts carry rounding of order 1e-7 |z| on an error of order |d|, so directions agree to ~1e-4; an
    # indexing mistake (direction paired with the wrong state) would give errors of order 1
    ok &= check(f"{t}: own-error directions = explicit model rollout (max abs)", float(np.abs(Ud - Uref).max()), 1e-2)
    Vr = np.asarray(fn["rand_dirs"](jax.random.PRNGKey(5), Xw))[0].reshape(H, -1)
    ok &= check(f"{t}: random probe directions are unit, none at k = 0", float(abs(np.linalg.norm(Vr[1:], axis=1) - 1).max() + np.abs(Vr[0]).max()), 1e-12)
    dd = (np.asarray(jax.vmap(m)(Xw[0])) - np.asarray(Yw[0])).astype(np.float64).reshape(H, -1)
    def ref_gap(Vs):
        l2, mc, b2, oc = [], [], [], []
        for k in range(1, H):
            a = JF[k] @ Vs[k]; b = JP[k] @ Vs[k]
            na, nb = np.linalg.norm(a), np.linalg.norm(b)
            l2.append(np.sum((a - b) ** 2) / D); b2.append(nb ** 2 / D)
            mc.append(((na - nb) ** 2 + BETA_COS * nb ** 2 * (1 - a @ b / (na * nb))) / D); oc.append(1 - a @ b / (na * nb))
        return np.mean(l2), np.mean(mc), np.mean(b2), np.mean(oc)
    ro, rc, rb, rcs = ref_gap(Ud); rr, _, rrb, _ = ref_gap(Vr)        # the same directions as passed to the terms
    tt = {k: float(v) for k, v in fn["terms"](m, Xw, Yw, jnp.asarray(Ud.reshape(1, H, nch, N)), jnp.asarray(Vr.reshape(1, H, nch, N))).items()}
    for k, rv, tol in (("mse", float(np.mean(dd ** 2)), 1e-6), ("own", ro, 1e-4), ("owncos", rc, 1e-4), ("own_b2", rb, 1e-8), ("cos", rcs, 1e-3),
                       ("rand", rr, 1e-4), ("rand_b2", rrb, 1e-8)):
        ok &= check(f"{t}: term '{k}' = explicit dense computation (rel)", abs(tt[k] - rv) / abs(rv), tol)
    # gradients, on a float64 copy of the network, with the probe directions held fixed (as in training)
    m64 = jax.tree_util.tree_map(lambda a: a.astype(jnp.float64) if eqx.is_array(a) else a, m)
    X64, Y64 = Xw.astype(jnp.float64), Yw.astype(jnp.float64)
    P0, static = eqx.partition(m64, eqx.is_array)
    Df = jax.lax.stop_gradient(fn["own_dirs"](m64, X64)); Dr = fn["rand_dirs"](jax.random.PRNGKey(5), X64)
    rdir = jax.tree_util.tree_map(lambda a: jnp.asarray(rng.normal(size=a.shape), a.dtype), P0)
    for term in ("own", "owncos", "rand", "cos"):
        def Lf(q):
            tq = fn["terms"](eqx.combine(q, static), X64, Y64, Df, Dr); return tq["mse"] + tq[term]
        g = eqx.filter_grad(Lf)(P0); gn = tnorm(g)
        for lab, dirn in (("along the gradient", g), ("random direction", rdir)):
            Lp = lambda s_: float(Lf(jax.tree_util.tree_map(lambda a, b: a + s_ * b, P0, dirn)))
            errs = []
            for k in range(5):     # central differences: a correct gradient's error falls ~16x per step, a wrong one stays
                hstep = 1e-5 * tnorm(P0) / tnorm(dirn) / 4 ** k
                errs.append(abs(tdot(g, dirn) - (Lp(hstep) - Lp(-hstep)) / (2 * hstep)) / (gn * tnorm(dirn)))
            print("       errors as the step shrinks 4x each time: " + "  ".join(f"{x:.1e}" for x in errs))
            ok &= check(f"{t}: gradient of 'mse + {term}' vs finite difference, {lab} (float64, best step)", min(errs), 1e-4)
    # the training objective's gradient = the gradient with the directions frozen at the current model (stop-gradient)
    PARTS = dict(own=("own",), owncos=("owncos",), both=("own", "rand"), cos=("cos",), randcos=("rand", "cos"))
    for kind in PARTS:
        g1 = eqx.filter_grad(lambda q: fn["objective"](eqx.combine(q, static), X64, Y64, jax.random.PRNGKey(5), 1.0, 1.0, 1.0, kind)[0])(P0)
        def Lfix(q):
            tq = fn["terms"](eqx.combine(q, static), X64, Y64, Df, Dr if kind in NEED_RAND else None)
            return tq["mse"] + sum(tq[p_] for p_ in PARTS[kind])
        g2 = eqx.filter_grad(Lfix)(P0)
        ok &= check(f"{t}: objective '{kind}': no gradient through the rollout direction (rel)", tnorm(jax.tree_util.tree_map(lambda a, b: a - b, g1, g2)) / tnorm(g2), 1e-10)
    if not SMOKE:
        for s in SEEDS[:1]:
            mm = load_dataset_model(t, s)
            e_, ef, eP, _ = [np.asarray(z) for z in fn["q1"](mm, jnp.asarray(TEST[t], jnp.float64))]
            fl = lambda z: z.reshape(z.shape[0], z.shape[1], -1)
            Nf = cc.first_above(cc.gmean(cc.split(fl(ef), fl(e_))["T"]), 0.25); NP = cc.first_above(cc.gmean(cc.split(fl(eP), fl(e_))["T"]), 0.25)
            rec = RESJ[f"{t}__{ARCH}__s{s}__best"]
            ok &= check(f"reproduce stored N*_f, N*_P of {t} {ARCH} s{s} ({Nf}/{NP} vs {rec['Nf']}/{rec['NP']})", abs(Nf - rec["Nf"]) + abs(NP - rec["NP"]), 2.5)
    if MODE == "scratch":
        same = all(bool(jnp.all(a == b)) for a, b in zip(jax.tree_util.tree_leaves(eqx.filter(init_model(t, 0), eqx.is_array)),
                                                         jax.tree_util.tree_leaves(eqx.filter(pc.make_net(ARCH, nch, jax.random.PRNGKey(0)), eqx.is_array))))
        ok &= check(f"{t}: scratch initialisation is the dataset's PRNGKey(seed) init (0 = same)", 0.0 if same else 1.0, 0.5)
print("VALIDATION", "ALL PASS" if ok else "FAILED: do not use the results"); assert ok

## Training

In [ ]:
def name(tag, seed, arm, ck=None):
    b = f"own__{MODE}__{tag}__{ARCH}__s{seed}__{arm}"
    return b if ck is None else f"{b}__{ck}"

RATES = {}                                           # measured updates/s per (tag, kind), to plan the session
class OutOfTime(Exception): pass

def train_run(tag, seed, arm, deadline):
    fn = FNS[tag]; kind, ratio = parse_arm(arm)
    net = load_dataset_model(tag, seed) if MODE == "finetune" else init_model(tag, seed)
    P0, static = eqx.partition(net, eqx.is_array)
    mk = lambda q: eqx.combine(q, static)
    tr = jnp.asarray(TRAIN[tag]); NT, T1 = tr.shape[0], tr.shape[1]
    total = UPDATES; t_on = int(TERM_ON * total); t_full = t_on + int(TERM_RAMP * total)
    every = max(total // N_EVAL, 10)
    base_key = jax.random.PRNGKey(10_000 + seed)

    def make_step(k):
        @eqx.filter_jit
        def tstep(q, ost, xw, yw, lr, it, lam_r, lam_o, lam_c):
            (lv, parts), g = eqx.filter_value_and_grad(lambda z: fn["objective"](mk(z), xw, yw, jax.random.fold_in(base_key, it), lam_r, lam_o, lam_c, k), has_aux=True)(q)
            up, nost = OPT.update(g, ost)
            up = jax.tree_util.tree_map(lambda u: u * lr, up)
            return eqx.apply_updates(q, up), nost, lv, parts["mse"]
        return tstep
    step_mse, step_term = make_step("mse"), make_step(kind)

    def lr_at(it):
        if it <= WARMUP: return LR * it / WARMUP
        return LR * (LR_MIN / LR + (1 - LR_MIN / LR) * 0.5 * (1 + math.cos(math.pi * (it - WARMUP) / max(total - WARMUP, 1))))

    rng = np.random.default_rng(1000 + seed)        # the same windows, in the same order, for every arm
    ar = np.arange(HW)
    def batch(r):
        ti = r.integers(0, NT, BATCH); si = r.integers(0, T1 - HW, BATCH); idx = si[:, None] + ar[None, :]
        return tr[ti[:, None], idx], tr[ti[:, None], idx + 1]

    lam = dict(r=0.0, o=0.0, c=0.0)
    def calibrate(q):
        r = np.random.default_rng(77 + seed); acc = {}
        for i in range(CAL_BATCHES):
            xw, yw = batch(r)
            for k_, v in fn["probe_terms"](mk(q), xw, yw, jax.random.PRNGKey(500 + i)).items():
                acc[k_] = acc.get(k_, 0.0) + float(v) / CAL_BATCHES
        if ratio["o"]: lam["o"] = ratio["o"] * acc["mse"] / acc["owncos" if kind == "owncos" else "own"]
        if ratio["r"]: lam["r"] = ratio["r"] * acc["mse"] / acc["rand"]
        if ratio["c"]: lam["c"] = ratio["c"] * acc["mse"] / acc["cos"]
        print(f"    calibration at update {it}: mse {acc['mse']:.3e} own {acc['own']:.3e} owncos {acc['owncos']:.3e} rand {acc['rand']:.3e}"
              f" 1-cos {acc['cos']:.3e} -> lam_rand {lam['r']:.3e} lam_own {lam['o']:.3e} lam_cos {lam['c']:.3e}", flush=True)
        return acc

    hist, best = [], dict(v=float("inf"), it=-1)
    def evaluate(q, it, lv, lm, ramp):
        mm = mk(q)
        X1, Y1 = VAL1[tag]
        os1 = float(fn["onestep"](mm, X1[:, 0], Y1[:, 0]))
        vr = float(fn["val_rollout"](mm, VALTRAJ[tag]))
        vt = val_terms(tag, mm)
        hist.append(dict(it=it, train_loss=lv, train_mse=lm, ramp=ramp, val_onestep_rel=os1, val_rollout=vr,
                         val_mse=vt["mse"], val_own_gap_rel=vt["own"] / vt["own_b2"], val_rand_gap_rel=vt["rand"] / vt["rand_b2"],
                         val_own_1mcos=vt["cos"]))
        eligible = (MODE == "finetune") or it >= t_full
        if eligible and np.isfinite(vr) and vr < best["v"]:
            best.update(v=vr, it=it); eqx.tree_serialise_leaves(os.path.join(CKD, name(tag, seed, arm, "best") + ".eqx"), mm)
        print(f"    [{tag}/s{seed}/{arm}] {it:>6} | one-step {os1:.5f} rollout(100) {vr:.4f} | gap/|J_P v| own {vt['own'] / vt['own_b2']:.3e}"
              f" rand {vt['rand'] / vt['rand_b2']:.3e} | best@{best['it']} {(time.time() - t_run) / 60:.1f}m", flush=True)

    t_run = time.time()
    p, ost = P0, OPT.init(eqx.filter(P0, eqx.is_array))
    it, cal, rate, stopped, t_cal = 0, None, float("nan"), "", None
    if kind != "mse" and t_on == 0:
        cal = calibrate(p)
    evaluate(p, 0, float("nan"), float("nan"), 0.0)
    while it < total:
        it += 1
        if kind != "mse" and it == t_on + 1 and t_on > 0:
            cal = calibrate(p)
        on = kind != "mse" and it > t_on
        ramp = 0.0 if not on else (1.0 if t_full <= t_on else min(1.0, (it - t_on) / (t_full - t_on)))
        xw, yw = batch(rng)
        f = step_term if on else step_mse
        p, ost, lv, lm = f(p, ost, xw, yw, jnp.asarray(lr_at(it), jnp.float32), jnp.asarray(it),
                           jnp.asarray(lam["r"] * ramp), jnp.asarray(lam["o"] * ramp), jnp.asarray(lam["c"] * ramp))
        if it == CAL0 + (t_on if on else 0):
            float(lv); t_cal = time.time()
        if t_cal is not None and it == CAL1 + (t_on if on else 0):
            float(lv); rate = (CAL1 - CAL0) / (time.time() - t_cal); t_cal = None
            RATES[(tag, kind)] = rate
            need = (total - it) / rate + RESERVE_MIN * 60
            if time.time() + need > deadline:
                raise OutOfTime(f"{rate:.1f} updates/s: {total - it} updates left need {need / 60:.0f} min, the session has "
                                f"{(deadline - time.time()) / 60:.0f} min")
            print(f"    {rate:.1f} updates/s ({'with' if on else 'without'} the term): about {(total - it) / rate / 60:.0f} min left", flush=True)
        if it % every == 0 or it == total:
            lf = float(lv)
            if not np.isfinite(lf):
                stopped = f"non-finite loss at {it}"; print("    " + stopped); break
            evaluate(p, it, lf, float(lm), ramp)
    if best["it"] < 0:                                 # nothing eligible was finite: keep the final model as "best"
        eqx.tree_serialise_leaves(os.path.join(CKD, name(tag, seed, arm, "best") + ".eqx"), mk(p))
    eqx.tree_serialise_leaves(os.path.join(CKD, name(tag, seed, arm, "final") + ".eqx"), mk(p))
    h = dict(hist=hist, updates=it, planned=total, rate=rate, best_it=best["it"], best_val_rollout=best["v"], stopped=stopped,
             arm=arm, kind=kind, ratio=ratio, lam_rand=lam["r"], lam_own=lam["o"], lam_cos=lam["c"], calibration=cal, t_on=t_on, t_full=t_full,
             mode=MODE, HW=HW, batch=BATCH, lr=LR, warmup=WARMUP, beta_cos=BETA_COS, minutes=(time.time() - t_run) / 60)
    json.dump(h, open(os.path.join(HIS, name(tag, seed, arm) + ".json"), "w"))
    return h

## Analysis on the 30 test trajectories

Rollout error, the two ledgers (model's Jacobian / solver's Jacobian, same mistakes), one-step mistake, the
Jacobian gap along the model's own rollout error and along random directions, the valid prediction time
(Vlachas et al. 2020: last step with NRMSE < 0.5, NRMSE normalised per channel by the training variance, in Lyapunov
times), the high-correlation times (last step with Pearson correlation above 0.9 / 0.8), and the leading Lyapunov
exponents of the model and the solver along 2 test trajectories.

In [ ]:
TESTPROBE = {}
for t in PDES:
    X = TEST[t][:, :S:TEST_EVERY].reshape((-1,) + TEST[t].shape[2:]).astype(np.float64)
    V = np.asarray(jax.random.normal(jax.random.PRNGKey(98), (X.shape[0], K_PROBES) + X.shape[1:], jnp.float64))
    V = V / np.sqrt(np.sum(V ** 2, axis=(-2, -1), keepdims=True))
    TESTPROBE[t] = (np.repeat(X, K_PROBES, 0), V.reshape((-1,) + X.shape[1:]))
def gap_stats(tag, model, X, V, ch=256):
    gb = [FNS[tag]["gap_along"](model, jnp.asarray(X[i:i + ch]), jnp.asarray(V[i:i + ch])) for i in range(0, len(X), ch)]
    if not gb:
        return np.zeros(0), np.zeros(0), np.zeros(0)
    return tuple(np.concatenate([np.asarray(x[j]) for x in gb]) for j in range(3))

JPD = {}
def analyze(tag, model, light=False):
    fn, nch, D = FNS[tag], CFG[tag]["nch"], FNS[tag]["D"]
    U = TEST[tag]; B = U.shape[0]
    e4, ef, eP, d = [np.asarray(z) for z in fn["q1"](model, jnp.asarray(U, jnp.float64))]
    e = e4.reshape(B, S, -1); ef = ef.reshape(B, S, -1); eP = eP.reshape(B, S, -1); d = d.reshape(B, S, -1)
    un = np.linalg.norm(U[:, 1:].reshape(B, S, -1).astype(np.float64), axis=-1)
    out = dict(nch=nch, S=S, lam=LAM[tag])
    with np.errstate(all="ignore"):
        out.update(rel_true=np.linalg.norm(e, axis=-1) / un, rel_f=np.linalg.norm(ef, axis=-1) / un, rel_P=np.linalg.norm(eP, axis=-1) / un)
        for nm, pr in (("f", ef), ("P", eP)):
            sp = cc.split(pr, e); out[f"T_{nm}"] = sp["T"]
        out["d_rel"] = np.linalg.norm(d, axis=-1) / un
        # valid prediction time and correlation times
        nrmse = np.sqrt(np.mean(e4 ** 2 / SIG2[tag][None, None, :, None], axis=(-2, -1)))
        pred = (e4 + U[:, 1:]).reshape(B, S, -1); tru = U[:, 1:].reshape(B, S, -1).astype(np.float64)
        pc_ = pred - pred.mean(-1, keepdims=True); tc_ = tru - tru.mean(-1, keepdims=True)
        corr = np.sum(pc_ * tc_, -1) / np.sqrt(np.sum(pc_ ** 2, -1) * np.sum(tc_ ** 2, -1))
    def last_ok(good):                                  # number of steps before the first failure (S if never)
        bad = ~good
        return np.array([int(np.argmax(bad[b])) if bad[b].any() else S for b in range(B)])
    out["nrmse"] = nrmse; out["corr"] = corr
    out["vpt_steps"] = last_ok(np.isfinite(nrmse) & (nrmse < VPT_THR))
    for th in CORR_THR:
        out[f"tcorr{int(th * 100)}_steps"] = last_ok(np.isfinite(corr) & (corr > th))
    bad = ~np.isfinite(out["rel_true"]) | (out["rel_true"] > 1e2)
    out["div_step"] = np.array([int(np.argmax(bad[b]) + 1) if bad[b].any() else S + 1 for b in range(B)])
    # the Jacobian gap along the model's own rollout error (at the true state u_n, error arriving there) and along random directions
    ns = np.arange(TEST_EVERY, S, TEST_EVERY)
    Xo = U[:, ns].reshape((-1, nch, N)).astype(np.float64)
    Eo = e4[:, ns - 1].reshape((-1, nch, N))
    nr = np.sqrt(np.sum(Eo ** 2, axis=(-2, -1), keepdims=True))
    okm = (np.isfinite(nr) & (nr > 0)).reshape(-1)
    Vo = np.where(okm[:, None, None], Eo / np.where(okm[:, None, None], nr, 1.0), 0.0)
    g, b2, oc = gap_stats(tag, model, Xo[okm], Vo[okm])
    out["own_gap"] = float(np.mean(g)) if g.size else float("nan"); out["own_gap_rel"] = float(np.sum(g) / np.sum(b2)) if g.size else float("nan")
    out["own_1mcos"] = float(np.mean(oc)) if g.size else float("nan")
    Xr, Vr = TESTPROBE[tag]; g, b2, _ = gap_stats(tag, model, Xr, Vr)
    out["rand_gap"] = float(np.mean(g)); out["rand_gap_rel"] = float(np.sum(g) / np.sum(b2))
    out = {k: (v.astype(np.float32) if isinstance(v, np.ndarray) and v.dtype == np.float64 else v) for k, v in out.items()}
    if light:
        return out
    if tag not in JPD:
        JPD[tag] = np.stack([np.concatenate([np.asarray(fn["jac_P"](jnp.asarray(U[b, i:min(i + CH, TDENSE)], jnp.float64).reshape(-1, D)))
                                             for i in range(0, TDENSE, CH)]) for b in range(NDENSE)])
    lyf, lyP = np.full((NDENSE, KLYAP), np.nan), np.full((NDENSE, KLYAP), np.nan)
    for b in range(NDENSE):
        JF = np.concatenate([np.asarray(fn["jac_f"](model, jnp.asarray(U[b, i:min(i + CH, TDENSE)], jnp.float64).reshape(-1, D))) for i in range(0, TDENSE, CH)])
        lyf[b] = cc.lyapunov(JF, KLYAP, burn=min(10, TDENSE // 4)); lyP[b] = cc.lyapunov(JPD[tag][b], KLYAP, burn=min(10, TDENSE // 4))
        del JF
    out.update(lyap_f=lyf, lyap_P=lyP)
    return out

In [ ]:
def done(n): return os.path.exists(os.path.join(RES, n + ".npz"))
def save(n, r): np.savez_compressed(os.path.join(RES, n + ".npz"), **r)
gm = lambda x: float(cc.gmean(np.asarray(x).reshape(-1)))
def line(r):
    g = cc.gmean(r["rel_true"])
    return (f"one-step {gm(r['d_rel']):.5f} | err@50/100/{S} {g[min(49, S - 1)]:.4f}/{g[min(99, S - 1)]:.4f}/{g[S - 1]:.4f} | "
            f"own gap {r['own_gap_rel']:.3e} (1-cos {r['own_1mcos']:.2e}) rand gap {r['rand_gap_rel']:.3e} | VPT {np.median(r['vpt_steps']) * r['lam']:.2f} LT | "
            f"diverged {int((r['div_step'] <= S).sum())}/{len(r['div_step'])}")

deadline = t_start + HOURS * 3600
for t in PDES:                                         # the dataset's one-step models, as a reference
    for s in SEEDS:
        n = name(t, s, "base", "best")
        if not done(n):
            r = analyze(t, load_dataset_model(t, s)); save(n, r); print(f"[base] {t} s{s}: {line(r)}", flush=True)

runs = [(t, s, arm) for t in PDES for s in SEEDS for arm in ARMS]
todo = [x for x in runs if not done(name(*x, "best"))]
print(f"{len(runs)} runs, {len(todo)} to do; session ends in {(deadline - time.time()) / 3600:.1f} h")
for i, (t, s, arm) in enumerate(todo):
    kind = parse_arm(arm)[0]
    est = RATES.get((t, kind))
    if est and time.time() + UPDATES / est + 2 * RESERVE_MIN * 60 > deadline:
        print(f"not enough time for {t} s{s} {arm} (about {UPDATES / est / 60:.0f} min): attach this output and run again to continue"); break
    print(f"\n[{i + 1}/{len(todo)}] {t} s{s} {arm}", flush=True)
    cks = os.path.join(CKD, name(t, s, arm, "best") + ".eqx")
    if not os.path.exists(cks) or not os.path.exists(os.path.join(HIS, name(t, s, arm) + ".json")):
        try:
            train_run(t, s, arm, deadline)
        except OutOfTime as ex:
            print(f"    stopped before finishing ({ex}); this run is redone next session")
            for ck in ("best", "final"):
                pth = os.path.join(CKD, name(t, s, arm, ck) + ".eqx")
                if os.path.exists(pth): os.remove(pth)
            break
    pf = os.path.join(CKD, name(t, s, arm, "final") + ".eqx")
    if os.path.exists(pf) and not done(name(t, s, arm, "final")):
        save(name(t, s, arm, "final"), analyze(t, eqx.tree_deserialise_leaves(pf, template(t)), light=True))
    res = analyze(t, eqx.tree_deserialise_leaves(cks, template(t)))
    save(name(t, s, arm, "best"), res)
    print(f"  best: {line(res)}", flush=True)

## Tables and figures

`table_own_summary.csv`: per mode, PDE and arm, medians over seeds; `*_vs_ctrl` = ratio to the `mse` arm of the same
seed (same data, same updates; below 1 = better); `*_vs_data` = ratio to the dataset's one-step model; `success` =
error below the control at steps 100 and 200 in every seed, with a one-step mistake no larger than the control's.
`table_own_runs.csv`: every run. Figures: `fig_own_error_<mode>.png` (error curves and per-seed ratios to the control),
`fig_own_tradeoff_<mode>.png` (one-step mistake vs error at step 200, both relative to the control),
`fig_own_training_<mode>.png` (validation curves). Attaching other sessions' outputs (either mode) gives the combined tables.

In [ ]:
rows, summ = oagg.aggregate(RES, HIS, OUT)
import zipfile
with zipfile.ZipFile(os.path.join(OUT, f"own_{MODE}_{'_'.join(PDES)}.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
    for p in (glob.glob(os.path.join(BASE, "**", "*"), recursive=True) + glob.glob(os.path.join(OUT, "fig_own_*.png"))
              + glob.glob(os.path.join(OUT, "table_own_*.csv"))):
        if os.path.isfile(p): zf.write(p, os.path.relpath(p, OUT))
print(f"done in {(time.time() - t_start) / 3600:.2f} h")